# P12 Environment and Control Checks Reproducible

由原 P12 拆出：整理 no-missing 消融索引、環境 proxy 範圍與補充控制檢查。


## 0. Setup


In [1]:
from __future__ import annotations

from pathlib import Path
from statistics import mean, median, stdev
from collections import Counter, defaultdict
from datetime import datetime
import csv
import json
import math
import re

try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except Exception:
    scipy_stats = None
    SCIPY_AVAILABLE = False

BASE = Path("/home/ester/valid_description")
TGT = BASE / "experiment data" / "journal_figures_tables_20260629"
FIG = TGT / "figures"
TAB = TGT / "tables"
DER = TGT / "derived_audits"
SRC = TGT / "source_programs"
REPO = BASE / "text-conditioned-outfit-recommendation"
EXP_MAIN = REPO / "experiments"
EXP_NONE = REPO / "experiments_none_only"

for p in [FIG, TAB, DER, SRC]:
    p.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
WORD_RE = re.compile(r"[A-Za-z]+(?:[-'][A-Za-z]+)?")

def read_json(path: Path):
    with path.open(encoding="utf-8") as f:
        return json.load(f)

def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        raise ValueError(f"No rows to write: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print("[write]", path)

def display_csv(path: Path, max_rows: int = 12) -> None:
    rows = read_csv(path)
    for r in rows[:max_rows]:
        print(r)
    if len(rows) > max_rows:
        print(f"... ({len(rows) - max_rows} more rows)")

def fmt_float(x, nd=4) -> str:
    try:
        if x is None or math.isnan(float(x)):
            return ""
        return f"{float(x):.{nd}f}"
    except Exception:
        return ""

def fmt_pct(num, den) -> str:
    return f"{100 * float(num) / float(den):.2f}" if den else "0.00"

def to_int(x, default=0) -> int:
    try:
        return int(float(x))
    except Exception:
        return default

def to_float(x, default=0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default

def query_token_count(text: str) -> int:
    return len(TOKEN_RE.findall(str(text or "").strip()))

def word_tokens(text: str) -> list[str]:
    return [t.lower() for t in WORD_RE.findall(str(text or "").strip())]

def word_count(text: str) -> int:
    return len(word_tokens(text))

def original_field(original_data: dict, set_id: str, field: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get(field, "") or "").strip()
    return str(value or "").strip()

def original_url_name_plus_title(original_data: dict, set_id: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        parts = [str(value.get("url_name", "") or "").strip(), str(value.get("title", "") or "").strip()]
        return " ".join(p for p in parts if p).strip()
    return str(value or "").strip()

def generated_title_only(generated_data: dict, set_id: str) -> str:
    value = generated_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get("title", "") or "").strip()
    return str(value or "").strip()

print("BASE =", BASE)
print("TAB  =", TAB)
print("DER  =", DER)
print("SCIPY_AVAILABLE =", SCIPY_AVAILABLE)


BASE = /home/ester/valid_description
TAB  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables
DER  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits
SCIPY_AVAILABLE = True


## 4. Existing No-Missing Ablation Artifacts


In [5]:
candidates = [
    (TAB / "T05_stage2_cp_ablation.csv", "table", "CP no-missing ablation: Full vs no-weather/no-occasion/no-style."),
    (TAB / "T06_stage2_cir_ablation.csv", "table", "CIR no-missing ablation: Full vs no-weather/no-occasion/no-style."),
    (TAB / "T07_stage2_factor_contribution_ratio.csv", "table", "Relative factor contribution ratios from existing no-missing ablation."),
    (TAB / "T10_stage2_cp_seed_detail.csv", "table", "Seed-level CP ablation detail."),
    (TAB / "T11_stage2_cir_seed_detail.csv", "table", "Seed-level CIR ablation detail."),
    (TAB / "T12_subset_robustness_summary.csv", "table", "Subset robustness summary from existing analysis."),
    (TAB / "T13_subset_delta_hit10_pivot.csv", "table", "Subset delta Hit@10 pivot from existing analysis."),
    (FIG / "F01_main_factor_contribution_proportion.png", "figure", "Existing factor contribution figure."),
    (FIG / "F02_subset_hitrate_comparison.png", "figure", "Existing subset Hit@10 comparison figure."),
    (FIG / "F03_subset_delta_hit10_heatmap.png", "figure", "Existing subset delta Hit@10 heatmap."),
    (FIG / "F04_subset_ablation_across_subsets.png", "figure", "Existing ablation-across-subsets figure."),
    (FIG / "F05_subset_per_dimension_bars.png", "figure", "Existing per-dimension subset bars."),
    (FIG / "F06_subset_rank_improvement_by_subset.png", "figure", "Existing rank-improvement-by-subset figure."),
]
a17 = []
for path, kind, use in candidates:
    a17.append({
        "artifact": path.name,
        "artifact_type": kind,
        "exists": str(path.exists()),
        "path": str(path),
        "source_scope": "experiments_none_only / existing subset and ablation analyses",
        "paper_use": use,
        "note": "Use as supplementary controlled-factor evidence; do not describe as a new no-image general rewrite baseline.",
    })
write_csv(DER / "A17_existing_no_missing_ablation_artifact_index.csv", a17)
display_csv(DER / "A17_existing_no_missing_ablation_artifact_index.csv", max_rows=20)


[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A17_existing_no_missing_ablation_artifact_index.csv
{'artifact': 'T05_stage2_cp_ablation.csv', 'artifact_type': 'table', 'exists': 'True', 'path': '/home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables/T05_stage2_cp_ablation.csv', 'source_scope': 'experiments_none_only / existing subset and ablation analyses', 'paper_use': 'CP no-missing ablation: Full vs no-weather/no-occasion/no-style.', 'note': 'Use as supplementary controlled-factor evidence; do not describe as a new no-image general rewrite baseline.'}
{'artifact': 'T06_stage2_cir_ablation.csv', 'artifact_type': 'table', 'exists': 'True', 'path': '/home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables/T06_stage2_cir_ablation.csv', 'source_scope': 'experiments_none_only / existing subset and ablation analyses', 'paper_use': 'CIR no-missing ablation: Full vs no-weather/no-o

## 6. Environment Proxy Range and Requested Controls


In [7]:
rel_rows = read_csv(REPO / "subset_analysis_results/reliability_meta_from_subset.csv")
temps, invalid_temp, missing_temp = [], 0, 0
for r in rel_rows:
    raw = str(r.get("temperature_c", "") or "").strip()
    if not raw:
        missing_temp += 1
        continue
    try:
        temps.append(float(raw))
    except Exception:
        invalid_temp += 1
def counts(col):
    return Counter(str(r.get(col, "") or "").strip() for r in rel_rows)
weather_counts = counts("weather_group")
occasion_counts = counts("occasion_group")
style_group_counts = counts("style_group")
style_counts, invalid_style = [], 0
for r in rel_rows:
    try:
        style_counts.append(int(float(str(r.get("style_count", "") or "").strip())))
    except Exception:
        invalid_style += 1
a16 = [
    {"proxy": "temperature_c", "n_rows": str(len(rel_rows)), "valid_count": str(len(temps)), "missing_count": str(missing_temp), "invalid_count": str(invalid_temp), "observed_range": f"{min(temps):.2f} to {max(temps):.2f}" if temps else "", "mean": fmt_float(mean(temps), 4) if temps else "", "out_of_reasonable_range_count": str(sum(1 for t in temps if t < -30 or t > 50)), "reasonable_range_rule": "-30 to 50 Celsius sanity range.", "error_handling_note": "Non-numeric or missing values are counted, not silently imputed."},
    {"proxy": "weather_group", "n_rows": str(len(rel_rows)), "valid_count": str(sum(weather_counts[g] for g in ["cold", "warm"])), "missing_count": str(weather_counts.get("", 0)), "invalid_count": str(len(rel_rows) - sum(weather_counts[g] for g in ["cold", "warm"]) - weather_counts.get("", 0)), "observed_range": "; ".join(f"{k}:{v}" for k, v in sorted(weather_counts.items())), "mean": "", "out_of_reasonable_range_count": "0", "reasonable_range_rule": "Allowed groups: cold, warm.", "error_handling_note": "Unexpected labels are counted as invalid."},
    {"proxy": "occasion_group", "n_rows": str(len(rel_rows)), "valid_count": str(sum(occasion_counts[g] for g in ["formal", "casual", "unknown"])), "missing_count": str(occasion_counts.get("", 0)), "invalid_count": str(len(rel_rows) - sum(occasion_counts[g] for g in ["formal", "casual", "unknown"]) - occasion_counts.get("", 0)), "observed_range": "; ".join(f"{k}:{v}" for k, v in sorted(occasion_counts.items())), "mean": "", "out_of_reasonable_range_count": "0", "reasonable_range_rule": "Allowed groups: formal, casual, unknown.", "error_handling_note": "Unknown is retained as an explicit category rather than guessed from images."},
    {"proxy": "style_count/style_group", "n_rows": str(len(rel_rows)), "valid_count": str(len(style_counts)), "missing_count": "0", "invalid_count": str(invalid_style), "observed_range": (f"style_count {min(style_counts)} to {max(style_counts)}; " if style_counts else "") + "; ".join(f"{k}:{v}" for k, v in sorted(style_group_counts.items())), "mean": fmt_float(mean(style_counts), 4) if style_counts else "", "out_of_reasonable_range_count": str(sum(1 for s in style_counts if s < 0 or s > 20)), "reasonable_range_rule": "Style term count must be non-negative.", "error_handling_note": "Numeric parsing failures are counted as invalid."},
]
write_csv(DER / "A16_environment_proxy_range_error_handling.csv", a16)

a15 = [
    {"control_question": "Main dataset/evaluation scope", "status": "completed", "evidence_file": str(TAB / "T00_dataset_overview_po_d_main.csv"), "paper_action": "Report PO-D as 16,995/3,000/15,145 = 35,140 outfits. Explain CIR 9,311 as task-specific candidate-pool eligibility, not CP reassignment."},
    {"control_question": "Whether improvement is merely caused by longer text", "status": "completed as diagnostic", "evidence_file": "; ".join([str(DER / "A08_length_performance_correlation_summary.csv"), str(DER / "A09_length_bucket_performance_summary.csv"), str(DER / "A10_matched_length_subset_summary.csv"), str(DER / "A11_length_defense_interpretation.txt")]), "paper_action": "Report length-effect size and p-value carefully. If p is significant only because N is large, emphasize negligible r and matched/bucket checks rather than claiming zero signal."},
    {"control_question": "Whether answer/target clues appear in generated query", "status": "completed as static lexical audit", "evidence_file": "; ".join([str(DER / "A03_target_item_clue_leakage_audit_summary.csv"), str(DER / "A04_target_item_clue_leakage_examples.csv")]), "paper_action": "Use leakage rate and examples as post-hoc validity check. Do not claim masked-query performance unless a masked CIR rerun is actually performed."},
    {"control_question": "Whether gains are just generic text rewriting", "status": "handled with existing controlled ablation artifacts; no new free rewrite baseline", "evidence_file": str(DER / "A17_existing_no_missing_ablation_artifact_index.csv"), "paper_action": "Use existing no-weather/no-occasion/no-style ablations as controlled factor-removal evidence. Explain that free no-image rewrite baseline is not used because it may hallucinate missing visual/item attributes and confound the research question."},
    {"control_question": "Statistical stability across seeds", "status": "completed", "evidence_file": str(DER / "A14_statistical_rigor_ci_adjusted_p_effect_sizes.csv"), "paper_action": "Report 5-seed mean/std, paired tests, CI, adjusted p-values, and effect sizes from main experiments after duplicate-row removal."},
    {"control_question": "Proxy ranges and error handling", "status": "completed", "evidence_file": str(DER / "A16_environment_proxy_range_error_handling.csv"), "paper_action": "Place detailed temperature/weather/occasion/style range checks in appendix; mention invalid/missing values are counted rather than silently imputed."},
    {"control_question": "LLM-as-a-Judge intervention proof", "status": "not needed for this P12 validity layer", "evidence_file": "", "paper_action": "State in method/evaluation text that LLM-as-a-Judge is post-hoc assessment only and does not rewrite, filter, train, or retrieve."},
]
write_csv(DER / "A15_requested_controls_status_summary.csv", a15)
display_csv(DER / "A16_environment_proxy_range_error_handling.csv")
display_csv(DER / "A15_requested_controls_status_summary.csv", max_rows=20)


[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A16_environment_proxy_range_error_handling.csv
[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A15_requested_controls_status_summary.csv
{'proxy': 'temperature_c', 'n_rows': '9311', 'valid_count': '9311', 'missing_count': '0', 'invalid_count': '0', 'observed_range': '-78.60 to 28.70', 'mean': '21.5311', 'out_of_reasonable_range_count': '9', 'reasonable_range_rule': '-30 to 50 Celsius sanity range.', 'error_handling_note': 'Non-numeric or missing values are counted, not silently imputed.'}
{'proxy': 'weather_group', 'n_rows': '9311', 'valid_count': '9311', 'missing_count': '0', 'invalid_count': '0', 'observed_range': 'cold:5264; warm:4047', 'mean': '', 'out_of_reasonable_range_count': '0', 'reasonable_range_rule': 'Allowed groups: cold, warm.', 'error_handling_note': 'Unexpected labels are counted as invalid.'}
{'proxy': 'occasion_grou

## Final Output Checks


In [8]:
expected = [
    TAB / "T00_dataset_overview_po_d_main.csv",
    TAB / "T01_text_field_comparison_po_d_main.csv",
    DER / "A01_title_length_audit.csv",
    DER / "A02_generation_coverage_audit.csv",
    DER / "A03_target_item_clue_leakage_audit_summary.csv",
    DER / "A04_target_item_clue_leakage_examples.csv",
    DER / "A07_length_performance_rowlevel_cir.csv",
    DER / "A08_length_performance_correlation_summary.csv",
    DER / "A09_length_bucket_performance_summary.csv",
    DER / "A10_matched_length_subset_summary.csv",
    DER / "A11_length_defense_interpretation.txt",
    DER / "A14_statistical_rigor_ci_adjusted_p_effect_sizes.csv",
    DER / "A15_requested_controls_status_summary.csv",
    DER / "A16_environment_proxy_range_error_handling.csv",
    DER / "A17_existing_no_missing_ablation_artifact_index.csv",
]
missing = []
for p in expected:
    if not p.exists() or p.stat().st_size == 0:
        missing.append(p)
    else:
        print("[ok]", p, p.stat().st_size)
if missing:
    raise FileNotFoundError("Missing or empty outputs:\n" + "\n".join(str(p) for p in missing))
a07 = read_csv(DER / "A07_length_performance_rowlevel_cir.csv")
print("A07 rows:", len(a07), "expected:", 9311 * 5)
if len(a07) != 9311 * 5:
    raise ValueError("A07 should use main 9,311 CIR pairs across 5 seeds, not the 3,432 no-missing subset.")
print("P12 completed with experiments/ as primary validity scope and experiments_none_only only as supplementary ablation index.")


[ok] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables/T00_dataset_overview_po_d_main.csv 1428
[ok] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables/T01_text_field_comparison_po_d_main.csv 702
[ok] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A01_title_length_audit.csv 634
[ok] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A02_generation_coverage_audit.csv 1674
[ok] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A03_target_item_clue_leakage_audit_summary.csv 753
[ok] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A04_target_item_clue_leakage_examples.csv 20738
[ok] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A07_length_performance_rowlevel_cir.csv 6107878
[ok] /home/ester/valid_descri